# Part 6 — Conclusions and Evidence Synthesis

**BFT65B — Big Data Analytics in Finance | Student ID: 11230945**

Consolidate verified results for Q1–Q3 and specify limitations and local adaptation requirements. All calculations use saved evidence. No historical test rescoring or new model selection is performed.

## 1. Repository and runtime
Use a fresh CPU runtime and the pinned environment. The complete offline test suite requires Java 17 or later.

In [2]:
from pathlib import Path
import os, sys, subprocess, json, shutil
REPO_URL = "https://github.com/NguyenDuong0508/11230945.git"
REPO = Path("/content/11230945")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    assert subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip() == REPO_URL
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
java = shutil.which("java")
if java is None:
    raise RuntimeError("Java 17 or later is required")
os.environ["JAVA_HOME"] = str(Path(java).resolve().parents[1])
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
print(subprocess.run([java,"-version"],check=True,capture_output=True,text=True).stderr)
subprocess.run([sys.executable,"-m","pip","install","-r",str(REPO/"requirements.txt")],check=True)
sys.path.insert(0,str(REPO/'src'))

Git revision: 2bc414045118e008369a9a3520d3a4ed7c2794af
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)



## 2. Mandatory offline validation
A failure or skipped test stops execution. Tests use synthetic fixtures; they do not access the historical test workspace.

In [3]:
import xml.etree.ElementTree as ET
TESTS_PASSED = False
result_path = Path('/content/part6_tests.xml')
result_path.unlink(missing_ok=True)
result = subprocess.run([sys.executable,'-m','pytest','-q','-ra','--tb=short','--show-capture=no',f'--junitxml={result_path}','tests'],cwd=REPO,env=dict(os.environ,PYTEST_DISABLE_PLUGIN_AUTOLOAD='1',SPARK_LOCAL_IP='127.0.0.1',PYSPARK_PYTHON=sys.executable,MPLBACKEND='Agg'),capture_output=True,text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
if not result_path.is_file(): raise RuntimeError('Missing pytest report')
cases=list(ET.parse(result_path).getroot().iter('testcase'))
problems=[(case,node) for case in cases for node in case if node.tag in ('failure','error','skipped')]
for case,node in problems:
    print(case.get('classname'),case.get('name'),node.tag,node.get('message'),node.text or '')
if result.returncode or not cases or problems: raise RuntimeError('Offline validation failed; inspect diagnostics')
TESTS_PASSED=True
print(f'Mandatory offline validation: PASS ({len(cases)} tests, zero skips)')

........................................................................ [ 50%]
......................................................................   [100%]
----------------- generated xml file: /content/part6_tests.xml -----------------
142 passed in 312.13s (0:05:12)

Mandatory offline validation: PASS (142 tests, zero skips)


## 3. Verified evidence chain and report-front-page quantities
Resolve all five active pointers and the completed modelling run. Check hashes, ancestry and aggregate reconciliations without decoding model inputs or scoring estimators. The full verification results remain in Notebooks 01–07.

In [5]:
from google.colab import drive
from fdic_ews.conclusions import collect
import pandas as pd
import numpy as np
from IPython.display import display, HTML, Markdown
from html import escape

drive.mount('/content/drive')
WORKSPACE = Path('/content/drive/MyDrive/11230945')
assert TESTS_PASSED
E = collect(WORKSPACE)
assert E['status'] == 'PASS'
M = E['modelling']
SELECTED = M['selected_model']

PRIMARY = '#365C72'

def table(data, title, formats=None):
    df = pd.DataFrame(data).copy()
    styler = (df.style.hide(axis='index').format(formats or {}, na_rep='—', escape='html')
              .set_table_styles([
                  {'selector':'th','props':[('background-color',PRIMARY),('color','white'),('padding','8px 10px'),('font-size','11px')]},
                  {'selector':'td','props':[('padding','7px 10px'),('color','#243746'),('border-bottom','1px solid #E2E6EA'),('font-size','11px')]},
                  {'selector':'tbody tr:nth-child(odd)','props':[('background-color','#F2F5F7')]},
                  {'selector':'tbody tr:nth-child(even)','props':[('background-color','white')]}]))
    display(HTML(f'<div style="margin:24px 0 30px;overflow-x:auto"><p><strong>{escape(title)}</strong></p>{styler.to_html()}</div>'))

source=E['source']
front=[('Acquisition method','FDIC BankFind API, quarter-by-quarter'),
       ('Acquisition started (UTC)',source['created_utc']),('Acquisition sealed (UTC)',source['sealed_utc']),
       ('Financial quarters',str(source['quarters'])),
       *[(k.title()+' rows',f'{v:,}') for k,v in source['row_counts'].items()],
       ('Distinct financial CERTs',f"{source['distinct_certs']:,}"),
       ('Development decision periods','1984Q1–2000Q4; 2002Q1–2006Q4'),
       ('Labelled development rows',f"{M['development_rows']:,}"),
       ('Distinct represented development events',f"{M['development_events']:,}"),
       ('Historical test decisions','2008Q1–2014Q4'),
       ('Labelled test rows',f"{int(M['test_metrics'][0]['rows']):,}"),
       ('Represented failures in test calendar',f"{E['features']['summary']['test_distinct_failure_events_in_calendar_period']:,}"),
       ('Represented failures through test horizons',f"{E['features']['summary']['test_distinct_positive_events_through_final_horizon']:,}")]
table(pd.DataFrame(front,columns=['Quantity','Value']), 'Table 1. Study scope and report-front-page evidence')
table([{'Stage':k.title(),'SHA-256':v} for k,v in E['lineage'].items()], 'Table 2. Verified evidence identities')
print('Evidence synthesis:',E['status'])
print('Verification scope:',E['verification_scope'])

Mounted at /content/drive


Quantity,Value
Acquisition method,"FDIC BankFind API, quarter-by-quarter"
Acquisition started (UTC),2026-11-25T17:36:55.524932+00:00
Acquisition sealed (UTC),2026-11-25T17:53:11.757758+00:00
Financial quarters,170
Financials rows,"1,682,739"
Failures rows,"4,117"
Institutions rows,"27,838"
Distinct financial CERTs,"24,064"
Development decision periods,1984Q1–2000Q4; 2002Q1–2006Q4
Labelled development rows,"1,082,584"


Stage,SHA-256
Bronze,e376152baac058e4dd4fff4582e9afa3c58820fff2f278115a82bff7fd4b3f6e
Silver,b6d91c7d0095b81e6200a0ae0940a103968c452046b168e446cd02089d323af0
Gold,de2abf748e29025760f50cc6c3c57d899a4402fe079756e17f9a6d93f21b5780
Storage,8516873cff244b6f943d2c282753093ef6ea2b6c64eba107b82decf951d9dc78
Processing,36ca1cbdd57b4fa74abc67fac85dafdb73d0080b1f73ea8f2aa086fbdeaf063f
Development,12532646188770d748eea1b494cb350410a23057887e552ba5568ec19c04072f
Test,8e6d5344d8a0bbb7fa56ff50be8f7b53796f89d6367cf8dc011ad72fae6298cd


Evidence synthesis: PASS
Verification scope: Artifact hashes, lineage and aggregate arithmetic; no model fitting or scoring


## 4. Q1 — Storage, processing and distributed computing
Compare equivalent workloads; speedups above one indicate lower median latency than the reference. File candidate counts are not physical bytes read. Report measured memory alongside latency, and do not infer a multi-node break-even from local Spark or synthetic scale-up.

In [6]:
S=E['storage']; P=E['processing']
table(S['sizes'],'Table 3. Storage footprint relative to CSV',{'mib':'{:,.2f}','reduction_pct':'{:.2f}'})
table(S['queries'],'Table 4. Storage workload comparison',{'seconds':'{:.4f}','speedup_vs_csv':'{:.2f}','peak_rss_mib':'{:,.1f}','candidate_fragments':'{:.0f}'})
proc=pd.DataFrame(P['results'])
table(proc[proc.scale.eq(1)],'Table 5. Original-panel processing performance',{'input_rows':'{:,.0f}','seconds':'{:.4f}','speedup_vs_pandas':'{:.2f}','time_ratio_vs_1x':'{:.2f}','peak_rss_mib':'{:,.1f}'})
table(proc[proc.scale.gt(1)],'Table 6. Synthetic scale-up performance',{'input_rows': '{:,.0f}','seconds': '{:.4f}','speedup_vs_pandas': '{:.2f}','time_ratio_vs_1x': '{:.2f}','peak_rss_mib': '{:,.1f}'})
ram=float(P['environment']['ram_mib'])
local=proc[proc.engine.eq('polars')].copy()
local['rss_to_host_ram_pct']=100*local.peak_rss_mib/ram
local=local[['workload','scale','seconds','peak_rss_mib','rss_to_host_ram_pct']]
table(local,'Table 7. Polars observed memory relative to recorded host RAM',{'seconds':'{:.4f}','peak_rss_mib':'{:,.1f}','rss_to_host_ram_pct':'{:.2f}'})
parallel=pd.DataFrame(P['parallelism'])
for col in parallel.columns.difference(['workload']): parallel[col]=pd.to_numeric(parallel[col])
table(parallel,'Table 8. One- versus two-thread Polars diagnostics',{c:'{:.4f}' for c in parallel.columns if c!='workload'})
print(f"Measured storage trials: {S['measured_trials']:,}; processing trials: {P['measured_trials']:,}; recorded host RAM: {ram:,.1f} MiB")
print('Smallest measured storage representation:',min(S['sizes'],key=lambda r:r['mib'])['format'])
for w,g in proc[proc.scale.eq(1)].groupby('workload'):
    winner=g.loc[g.seconds.idxmin()]
    print(f"{w}: lowest observed median latency = {winner.engine} ({winner.seconds:.4f} s)")

format,mib,reduction_pct
csv,354.45,0.00
flat_snappy,211.79,40.25
flat_zstd,178.60,49.61
partitioned_snappy,243.43,31.32
partitioned_zstd,192.02,45.82


format,workload,seconds,speedup_vs_csv,peak_rss_mib,candidate_fragments
csv,full,4.0164,1.00,"4,161.7",1
flat_snappy,full,1.1430,3.51,"4,161.7",1
flat_zstd,full,1.2801,3.14,"4,161.7",1
partitioned_snappy,full,1.4919,2.69,"4,161.7",170
partitioned_zstd,full,1.6436,2.44,"4,161.7",170
csv,projection,1.9802,1.00,"4,161.7",1
flat_snappy,projection,0.2281,8.68,"4,161.7",1
flat_zstd,projection,0.2145,9.23,"4,161.7",1
partitioned_snappy,projection,0.4114,4.81,"4,161.7",170
partitioned_zstd,projection,0.4128,4.80,"4,161.7",170


workload,scale,engine,input_rows,seconds,speedup_vs_pandas,time_ratio_vs_1x,peak_rss_mib
quarter_summary,1,pandas,"751,323",0.3337,1.00,1.00,310.9
quarter_summary,1,polars,"751,323",0.0874,3.82,1.00,123.3
quarter_summary,1,pyspark,"751,323",6.3112,0.05,1.00,795.8
adjacent_growth,1,pandas,"1,682,739",0.8028,1.00,1.00,490.2
adjacent_growth,1,polars,"1,682,739",0.6814,1.18,1.00,253.8
adjacent_growth,1,pyspark,"1,682,739",4.9578,0.16,1.00,"1,067.7"
peer_join,1,pandas,"1,682,739",1.6625,1.00,1.00,478.2
peer_join,1,polars,"1,682,739",0.3162,5.26,1.00,326.1
peer_join,1,pyspark,"1,682,739",5.6767,0.29,1.00,968.7


workload,scale,engine,input_rows,seconds,speedup_vs_pandas,time_ratio_vs_1x,peak_rss_mib
quarter_summary,5,pandas,"3,756,615",1.1402,1.00,3.42,819.3
quarter_summary,5,polars,"3,756,615",0.2027,5.62,2.32,322.9
quarter_summary,10,pandas,"7,513,230",2.6675,1.00,7.99,"1,430.4"
quarter_summary,10,polars,"7,513,230",0.3500,7.62,4.00,587.0
adjacent_growth,5,pandas,"8,413,695",5.1444,1.00,6.41,"1,672.0"
adjacent_growth,5,polars,"8,413,695",3.5006,1.47,5.14,909.9
adjacent_growth,10,pandas,"16,827,390",9.6139,1.00,11.98,"3,068.7"
adjacent_growth,10,polars,"16,827,390",7.9352,1.21,11.64,"1,700.7"
peer_join,5,pandas,"8,413,695",7.8075,1.00,4.70,"1,740.9"
peer_join,5,polars,"8,413,695",2.4692,3.16,7.81,"1,519.1"


workload,scale,seconds,peak_rss_mib,rss_to_host_ram_pct
quarter_summary,1,0.0874,123.3,0.95
quarter_summary,5,0.2027,322.9,2.49
quarter_summary,10,0.3500,587.0,4.52
adjacent_growth,1,0.6814,253.8,1.96
adjacent_growth,5,3.5006,909.9,7.01
adjacent_growth,10,7.9352,"1,700.7",13.11
peer_join,1,0.3162,326.1,2.51
peer_join,5,2.4692,"1,519.1",11.71
peer_join,10,3.2485,"3,079.0",23.73


workload,one_thread_s,two_threads_s,speedup,efficiency,apparent_serial_fraction
quarter_summary,0.1145,0.0874,1.3093,0.6547,0.5275
adjacent_growth,1.2192,0.6814,1.7891,0.8945,0.1179
peer_join,0.3964,0.3162,1.2534,0.6267,0.5957


Measured storage trials: 110; processing trials: 72; recorded host RAM: 12,975.5 MiB
Smallest measured storage representation: flat_zstd
adjacent_growth: lowest observed median latency = polars (0.6814 s)
peer_join: lowest observed median latency = polars (0.3162 s)
quarter_summary: lowest observed median latency = polars (0.0874 s)


**Interpretation boundary.** Use the measured workload winners and the full latency/size trade-offs rather than assuming a single format is uniformly best. Host RAM is context, not an available-memory or out-of-memory guarantee. Benchmark memory definitions and timing boundaries differ between Parts 1 and 3. The original Bronze layout remains immutable. A cluster requires evidence on service deadlines, concurrent workloads, data locality, memory pressure and network/provisioning cost; actual multi-node break-even has not been measured.

## 5. Q2 — Data quality and temporal integrity
Separate source-report problems from outcome adjudication. Count flagged observations, not unique institutions. Flags may overlap. Source availability sensitivity is not a model sensitivity experiment.

In [7]:
Q=E['quality']; G=E['features']; ss=Q['summary']
table([{'Diagnostic':k,'Value':f'{ss[k]:,}'} for k in ['source_rows','panel_rows','rows_removed','asset_to_null','gap_rows','event_ledger_rows','distinct_certs']], 'Table 9. Silver accounting')
fields=pd.DataFrame(Q['fields'])
for c in ['missing_pct','zero_pct']: fields[c]=pd.to_numeric(fields[c])
table(fields[['field','missing_pct','zero_pct']].sort_values('missing_pct',ascending=False).head(12),'Table 10. Highest field-level missingness',{'missing_pct':'{:.2f}','zero_pct':'{:.2f}'})
table(Q['flags'],'Table 11. Quality flags',{'flagged_rows':'{:,.0f}'})
table(Q['terminal'],'Table 12. Retrospective terminal audit (not a modelling filter)')
events=ss['event_diagnostics']
table([{'Diagnostic':k,'Records or events':v} for k,v in events.items() if isinstance(v,int)],'Table 13. Event linkage and classification evidence',{'Records or events':'{:,.0f}'})
a=pd.DataFrame(Q['availability'])
a['quarter_offset']=(pd.PeriodIndex(a.decision_quarter,freq='Q').asi8-pd.PeriodIndex(a.report_quarter,freq='Q').asi8)
for c in ['lag_days','available_rows','report_rows']: a[c]=pd.to_numeric(a[c])
a=a.groupby(['lag_days','quarter_offset'],as_index=False)[['available_rows','report_rows']].sum()
a['available_pct']=100*a.available_rows/a.report_rows
table(a,'Table 14. Source-report availability by assumed lag and decision-quarter offset',{'available_rows':'{:,.0f}','report_rows':'{:,.0f}','available_pct':'{:.2f}'})
sp=pd.DataFrame(G['splits'])
for c in ['risk_rows','labelled','positive','negative','censored']: sp[c]=pd.to_numeric(sp[c])
sp['censored_pct']=100*sp.censored/sp.risk_rows.replace(0,np.nan)
table(sp[['split','risk_rows','labelled','positive','negative','censored','censored_pct']],'Table 15. Gold risk and outcome accounting',{**{c:'{:,.0f}' for c in ['risk_rows','labelled','positive','negative','censored']},'censored_pct':'{:.2f}'})
print('Horizon-invariance diagnostic:',ss['horizon_case'])
print('Gold information-availability policy:',G['policy'])

Diagnostic,Value
source_rows,"1,682,739"
panel_rows,"1,682,739"
rows_removed,0
asset_to_null,0
gap_rows,33
event_ledger_rows,"31,955"
distinct_certs,"24,064"


field,missing_pct,zero_pct
P3ASSET,57.77,3.71
NTLNLS,4.75,12.28
P9ASSET,4.62,30.38
NAASSET,4.62,19.23
ELNATR,2.57,20.20
ROE,1.11,0.11
BRO,1.00,78.53
EINTEXP,0.38,0.82
NETINC,0.28,0.11
INTINC,0.28,0.05


rule,flagged_rows
ASSET_NONPOSITIVE,0
CAPITAL_ZERO,"468,492"
DEP_GT_ASSET,"6,205"
LOANS_GT_ASSET,763
NEGATIVE_DEP,0
NEGATIVE_EQUITY,"15,647"
NEGATIVE_LOANS,33
NONCURRENT_GT_LOANS,38
NONFINITE_VALUE,0
REPORTING_GAP,33


audit_class,institutions
dated_other_end_candidate,16711
end_precedes_last_report,3
failure_evidence,2570
failure_precedes_last_report,266
multiple_failure_dates,3
observed_at_study_end,4314
unresolved_discontinuation,197


Diagnostic,Records or events
assistance_records,593
duplicated_fin_values,61
end_dates_after_study_end,"4,284"
failure_dates_before_first_report,2
failure_dates_before_last_report,272
failure_type_records,"3,524"
invalid_or_missing_failure_dates,0
linked_distinct_failure_cert_dates_through_study_end,"2,842"
linked_distinct_failure_certs_through_study_end,"2,839"
missing_cert_failure_source_records,488


lag_days,quarter_offset,available_rows,report_rows,available_pct
60,0,0,"1,682,739",0.00
60,1,"1,682,739","1,682,739",100.00
60,2,"1,682,739","1,682,739",100.00
90,0,0,"1,682,739",0.00
90,1,"1,682,739","1,682,739",100.00
90,2,"1,682,739","1,682,739",100.00
120,0,0,"1,682,739",0.00
120,1,0,"1,682,739",0.00
120,2,"1,682,739","1,682,739",100.00
180,0,0,"1,682,739",0.00


split,risk_rows,labelled,positive,negative,censored,censored_pct
purge_development_test,"34,581","33,303",49,"33,254","1,278",3.70
purge_train_validation,"39,223","37,597",39,"37,558","1,626",4.15
test,"212,570","205,672","2,007","203,665","6,898",3.25
train,"948,824","906,299","7,718","898,581","42,525",4.48
unused_post_test,"222,835","214,484",132,"214,352","8,351",3.75
validation,"182,958","176,285",45,"176,240","6,673",3.65


Horizon-invariance diagnostic: {'CERT': 7213, 'full_result': [0, 'observed_at_horizon'], 'horizon_end': '2007-12-31', 'invariant': True, 'later_assistance_count': 1, 'observed_report': True, 'report_quarter': '2006-12-31', 'truncated_result': [0, 'observed_at_horizon']}
Gold information-availability policy: {'development_end': '2006-12-31', 'historical_vintages_verified': False, 'lag_days': 90, 'last_decision': '2025-06-30', 'max_report_age_quarters': 2, 'minimum_test_failure_certs': 50, 'seed': 11230945, 'test_end': '2014-12-31', 'test_start': '2008-03-31', 'train_end': '2000-12-31', 'validation_start': '2002-03-31'}


**Interpretation boundary.** No future terminal class selects historical observations. Failures and assistance are distinct; an uncorroborated end is not a confirmed merger. Reported capital zeros and conditional YTD income conversions remain qualified. The assumed reporting lag and retrospective FDIC snapshot do not prove true historical availability. The labelled population excludes unresolved outcomes, which may induce selection effects.

## 6. Q3 — Historical early-warning feasibility
The model and economic threshold remain frozen. Ranking, operating performance and distinct event detection answer different questions. The calendar population is smaller than the full forecast-horizon population. Annual event counts cannot be added.

In [8]:
metrics=pd.DataFrame(M['test_metrics']); calendar=pd.DataFrame(M['calendar'])
cols=['rows','positives','prevalence','ap','roc_auc','brier','log_loss','recall_top5pct_by_quarter','precision','recall','fpr','alert_rate','threshold','alerts','fp','events','detected_events','missed_events','event_recall','median_lead_days','lead_at_least_90_days','loss_usd_thousands']
for c in cols: metrics[c]=pd.to_numeric(metrics[c],errors='coerce')
for c in ['represented_events','distinct_certs','detected_events','event_recall','median_lead_days']: calendar[c]=pd.to_numeric(calendar[c],errors='coerce')
pct={c:'{:.2%}' for c in ['prevalence','recall_top5pct_by_quarter','precision','recall','fpr','alert_rate','event_recall']}
rank=['model','rows','positives','prevalence','ap','roc_auc','brier','log_loss','recall_top5pct_by_quarter']
table(metrics[rank],'Table 16. Historical test discrimination and probability error',{**pct,**{c:'{:.6f}' for c in ['ap','roc_auc','brier','log_loss']},'rows':'{:,.0f}','positives':'{:,.0f}'})
table(metrics[['model','threshold','alerts','fp','precision','recall','fpr','alert_rate']],'Table 17. Frozen operating policy',{**pct,'threshold':'{:.9f}','alerts':'{:,.0f}','fp':'{:,.0f}'})
table(metrics[['model','events','detected_events','missed_events','event_recall','median_lead_days','lead_at_least_90_days']],'Table 18. Failure-event detection through the test horizons',{**pct,'median_lead_days':'{:.1f}'})
table(calendar,'Table 19. Represented failures within the test calendar',pct)
refs=pd.DataFrame(M['reference_policies'])[['policy','fp','missed_events','loss_usd_thousands']].rename(columns={'policy':'Policy'})
sel=metrics[metrics.model.eq(SELECTED)][['model','fp','missed_events','loss_usd_thousands']].rename(columns={'model':'Policy'})
loss=pd.concat([sel,refs],ignore_index=True)
for c in ['fp','missed_events','loss_usd_thousands']: loss[c]=pd.to_numeric(loss[c])
loss['loss_usd_million']=loss.pop('loss_usd_thousands')/1000
table(loss,'Table 20. Economic scenario comparison (nominal USD millions)',{'fp':'{:,.0f}','missed_events':'{:,.0f}','loss_usd_million':'{:,.3f}'})
table([{'Diagnostic':k,'Value':v} for k,v in M['derived'].items()],'Table 21. Derived research indicators',{'Value':'{:.4f}'})
annual=pd.DataFrame(M['annual']); annual=annual[annual.model.eq(SELECTED)].copy()
for c in ['year','prevalence','ap','roc_auc','recall','event_recall']: annual[c]=pd.to_numeric(annual[c],errors='coerce')
table(annual[['year','prevalence','ap','roc_auc','recall','event_recall']],'Table 22. Annual decision-period stability',{**pct,'ap':'{:.4f}','roc_auc':'{:.4f}','year':'{:.0f}'})
weak=annual.loc[annual.event_recall.idxmin()]
print(f"Lowest annual event recall: {int(weak.year)} ({weak.event_recall:.2%}); AP in that year: {weak.ap:.4f}")
print('Selected development model:',SELECTED,'| mean fold AP:',M['selected_mean_ap'])
print('Frozen thresholds:',M['thresholds'])
print('Economic cost source:',M['cost']['missed_cost_source'])
print('Eligible dated FDIC cost events:',M['cost']['eligible_dated_cost_events'])
print('Spark validation AP:',M['spark']['ap'],'| validation sampled:',M['spark']['validation_sampled'])

model,rows,positives,prevalence,ap,roc_auc,brier,log_loss,recall_top5pct_by_quarter
lightgbm_no_quality,"205,672","2,007",0.98%,0.468612,0.975284,0.006909,0.027616,87.89%
prior,"205,672","2,007",0.98%,0.009758,0.500000,0.009670,0.055310,1.89%


model,threshold,alerts,fp,precision,recall,fpr,alert_rate
lightgbm_no_quality,0.033673132,"4,738","3,236",31.70%,74.84%,1.59%,2.30%
prior,2.000000000,0,0,—,0.00%,0.00%,0.00%


model,events,detected_events,missed_events,event_recall,median_lead_days,lead_at_least_90_days
lightgbm_no_quality,513,482,31,93.96%,282.5,436
prior,513,0,513,0.00%,—,0


model,represented_events,distinct_certs,detected_events,event_recall,median_lead_days
lightgbm_no_quality,505,505,475,94.06%,282.000000
prior,505,505,0,0.00%,—


Policy,fp,missed_events,loss_usd_million
lightgbm_no_quality,"3,236",31,331.937
no_alarm,0,513,"2,268.742"
all_alarm,"203,665",0,"12,262.670"


Diagnostic,Value
ap_lift_vs_prevalence,48.0221
event_recall_pct,93.9571
calendar_event_recall_pct,94.0594
early_90d_all_events_pct,84.9903
loss_reduction_vs_prior_pct,85.3691
loss_reduction_vs_no_alarm_pct,85.3691


year,prevalence,ap,roc_auc,recall,event_recall
2008,1.08%,0.3902,0.9541,34.73%,47.24%
2009,2.09%,0.6356,0.9718,71.49%,87.32%
2010,1.49%,0.6006,0.9811,90.15%,95.67%
2011,0.91%,0.5149,0.9917,95.17%,99.15%
2012,0.46%,0.4178,0.9952,97.67%,100.00%
2013,0.29%,0.4011,0.9825,81.01%,92.11%
2014,0.18%,0.3587,0.9934,89.13%,90.48%


Lowest annual event recall: 2008 (47.24%); AP in that year: 0.3902
Selected development model: lightgbm_no_quality | mean fold AP: 0.21677838466200192
Frozen thresholds: {'lightgbm_no_quality': 0.03367313153323793, 'prior': 2.0}
Economic cost source: assumed_10pct_of_median_failed_bank_assets
Eligible dated FDIC cost events: 0
Spark validation AP: 0.27106720327233425 | validation sampled: False


**Interpretation boundary.** Top-5% recall is a positive-row ranking measure on labelled observations. Failure-event detection is not causal prevention; lead time depends on assumed decision-time availability. The economic scenario is based on frozen cost assumptions, not realized savings. Low annual operating recall can coexist with strong discrimination. The historical holdout provides an out-of-time evaluation under the sealed chronological protocol. The retrospective results do not establish prospective performance, and post-test diagnostics do not change the locked policy.

## 7. Research limitations and adaptation to Vietnam
The following matrix defines requirements for a future local pilot, not empirical findings about Vietnamese banks. No US score or threshold is transferred. See `docs/part6_protocol.md` for sources and interpretation boundaries.

In [9]:
limitations=[
('Information availability','Assumed publication lag; historical revisions unobserved','Versioned submissions with actual release and revision timestamps'),
('Event and accounting semantics','Unlinked events, uncertain exits, ambiguous capital zeros and historical income conventions','Local reporting dictionary and dated, independently adjudicated events'),
('Evaluation population','Censored rows; repeated decision windows; retrospective crisis-period holdout','Quantify censoring and event-level uncertainty; independently held-out or prospective validation'),
('Economic decision','Asset-scaled proxy costs; fixed threshold may transfer poorly','Estimate review capacity and local costs; lock a policy on local development'),
('Scale','Single-node timings and synthetic replication; no multi-node break-even','Measure concurrent workloads, memory, deadlines, network and cost before cluster adoption'),
('Governance','Research demonstration; no operational deployment claim','Authorized access, human review, audit trails and ongoing drift monitoring')]
table(pd.DataFrame(limitations,columns=['Domain','Observed limitation','Required next evidence']),'Table 23. Limitations and local adaptation requirements')
print('Part 6 synthesis: PASS')
print('Historical test evidence remains closed; no models scored.')
print('Development seal:',E['lineage']['development'])
print('Test seal:',E['lineage']['test'])

Domain,Observed limitation,Required next evidence
Information availability,Assumed publication lag; historical revisions unobserved,Versioned submissions with actual release and revision timestamps
Event and accounting semantics,"Unlinked events, uncertain exits, ambiguous capital zeros and historical income conventions","Local reporting dictionary and dated, independently adjudicated events"
Evaluation population,Censored rows; repeated decision windows; retrospective crisis-period holdout,Quantify censoring and event-level uncertainty; independently held-out or prospective validation
Economic decision,Asset-scaled proxy costs; fixed threshold may transfer poorly,Estimate review capacity and local costs; lock a policy on local development
Scale,Single-node timings and synthetic replication; no multi-node break-even,"Measure concurrent workloads, memory, deadlines, network and cost before cluster adoption"
Governance,Research demonstration; no operational deployment claim,"Authorized access, human review, audit trails and ongoing drift monitoring"


Part 6 synthesis: PASS
Historical test evidence remains closed; no models scored.
Development seal: 12532646188770d748eea1b494cb350410a23057887e552ba5568ec19c04072f
Test seal: 8e6d5344d8a0bbb7fa56ff50be8f7b53796f89d6367cf8dc011ad72fae6298cd


### References and report handoff
- BFT65B examination instructions, Parts 1–5 protocols and executed evidence.
- [BCBS risk-data aggregation principles](https://www.bis.org/publications/201301-guidelines-principles-effective-risk-data-aggregation-and-risk-reporting) and [Core Principles for effective banking supervision](https://www.bis.org/committees/bcbs/basel-framework/standard/bcp?allChapters=true).
- [Vietnam Law on Credit Institutions No. 32/2024/QH15](https://vanban.chinhphu.vn/?classid=1&docid=211190&pageid=27160&typegroupid=3) and [amending Law No. 96/2025/QH15](https://vanban.chinhphu.vn/?classid=1&docid=214588&pageid=27160&typegroupid=3); confirm subsequent amendments and applicable rules before local deployment.

Save the executed notebook with outputs. Use the evidence to answer Q1–Q3 in the report and identify limitations and local requirements.